# 🧬 NanoSquiggle AMR: Pipeline em Cascata (2 Fases)
Este notebook faz o fluxo completo da nova arquitetura em cascata:
1. **Fase 1 (Binária):** Detetar se há gene (1) ou apenas ruído (0).
2. **Fase 2 (Multiclasse):** Dos genes detetados, identificar qual é (0, 1 ou 2).

In [ ]:
# --- 1. PREPARAÇÃO DO AMBIENTE ---
from google.colab import drive
import os

drive.mount('/content/drive')

!pip install pod5 pysam pandas scikit-learn matplotlib

os.chdir('/content')
!rm -rf NanoSquiggle-AMR-CNN
!git clone https://github.com/martinzx13/NanoSquiggle-AMR-CNN.git
os.chdir('NanoSquiggle-AMR-CNN')

print("✅ Ambiente preparado!")

In [ ]:
# --- 2. EXECUÇÃO DO DORADO ---
DORADO_BIN = "/content/drive/MyDrive/Klebsiella_POD5/dorado-0.5.3-linux-x64/bin/dorado"
POD5_DIR = "/content/drive/MyDrive/Raw_Data/KP1779"
REFERENCE = "data/raw/db_resistencia.fasta"
OUTPUT_SAM = "aligned_reads.sam"

print("🧬 A executar Dorado Basecaller...")
!chmod +x {DORADO_BIN}
!{DORADO_BIN} basecaller hac {POD5_DIR} --reference {REFERENCE} --emit-moves --emit-sam > {OUTPUT_SAM}
print("✅ Alinhamento concluído!")

In [ ]:
# --- 3. GERAÇÃO DOS DATASETS EM CASCATA ---
print("📊 A gerar os ficheiros CSV para as Fases 1 e 2...")
!python scripts/generate_cascade_datasets.py --sam {OUTPUT_SAM} --pod5_dir {POD5_DIR} --out_binary data/cascade_binary.csv --out_multi data/cascade_multi.csv


In [ ]:
# --- 4. TREINO DOS MODELOS EM CASCATA ---
print("🚀 A iniciar o treino dos modelos Binary e Multiclass...")
!python scripts/train_cascade.py --binary_csv data/cascade_binary.csv --multi_csv data/cascade_multi.csv --epochs 10 --batch_size 32
